<a href="https://colab.research.google.com/github/Md-Istiaq/Pre_Trained_Baseline_model-XLM_RoBERTa-_implementation/blob/main/Pre_Trained_Baseline_model(XLM_RoBERTa)_implementation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

pre-trained baseline model

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
import torch

# Load the dataset
df = pd.read_excel('Balanced_BanglaSpamData.xlsx')

# Convert labels to numeric format
label_mapping = {'spam': 1, 'ham': 0}
df['label'] = df['v1'].map(label_mapping)

# Create a smaller subset for faster execution (remove this for full dataset)
df_subset = df.sample(frac=0.1, random_state=42)

# Split data into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_subset['v2'].tolist(),
    df_subset['label'].tolist(),
    test_size=0.2,
    random_state=42
)

# Load pre-trained tokenizer and model
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)

# Tokenize the texts with padding to ensure consistent length
train_encodings = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, return_tensors="pt")

# Access input_ids and attention_mask directly from the encodings
train_input_ids = train_encodings['input_ids']
train_attention_mask = train_encodings['attention_mask']
train_labels = torch.tensor(train_labels)

test_input_ids = test_encodings['input_ids']
test_attention_mask = test_encodings['attention_mask']
test_labels = torch.tensor(test_labels)

# Make predictions (no training)
with torch.no_grad():
    outputs = model(input_ids=test_input_ids, attention_mask=test_attention_mask)
    predictions = outputs.logits.argmax(-1)

# Evaluate the model
accuracy = accuracy_score(test_labels, predictions)
precision = precision_score(test_labels, predictions, average='weighted', zero_division=1)
recall = recall_score(test_labels, predictions, average='weighted', zero_division=1)
f1 = f1_score(test_labels, predictions, average='weighted', zero_division=1)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy: 0.4
Precision: 0.76
Recall: 0.4
F1 Score: 0.22857142857142856


In [ ]:


import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# Load the dataset
df = pd.read_excel('Balanced_BanglaSpamData.xlsx')
df.columns = ['v1', 'v2']

# Encode labels
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

# Split data into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['v2'].tolist(),
    df['v1'].tolist(),
    test_size=0.2,
    random_state=42
)

# Load pre-trained tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)

# Tokenize the texts with padding to ensure consistent length
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")


# Create PyTorch datasets
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['v1'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
test_dataset = SpamDataset(test_encodings, test_labels)


# Make predictions (no training - Baseline)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()
predictions = []

with torch.no_grad():
  for batch in torch.utils.data.DataLoader(test_dataset, batch_size=16): # Adjust batch size as needed
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      outputs = model(input_ids, attention_mask=attention_mask)
      logits = outputs.logits
      preds = logits.argmax(-1).cpu().numpy()
      predictions.extend(preds)


# Evaluate the model
accuracy = accuracy_score(test_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, predictions, average='weighted', zero_division=1)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-13-c553aa66637b>:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Accuracy: 0.4941860465116279
Precision: 0.7500338020551649
Recall: 0.4941860465116279
F1 Score: 0.32689349380146593


train-test split the split uses the original dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# Load the dataset
df = pd.read_excel('Balanced_BanglaSpamData.xlsx')
df.columns = ['v1', 'v2']

# Encode labels
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

# Split the original dataset into training and testing sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['v2'].tolist(),  # Original text data
    df['v1'].tolist(),  # Original labels
    test_size=0.2,      # Use 20% of the original data for testing
    random_state=42     # Ensure reproducibility
)

# Load pre-trained tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)

# Tokenize the training and testing texts with padding and truncation
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

# Create PyTorch datasets
class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
test_dataset = SpamDataset(test_encodings, test_labels)

# Make predictions (no training - Baseline)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

predictions = []
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_dataset, batch_size=16):  # Adjust batch size as needed
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = logits.argmax(-1).cpu().numpy()
        predictions.extend(preds)

# Evaluate the model
accuracy = accuracy_score(test_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, predictions, average='weighted', zero_division=1)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-2-cdbd7e9aab9c>:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Accuracy: 0.491566265060241
Precision: 0.7500711278850342
Recall: 0.491566265060241
F1 Score: 0.3240049049185433
